# TerraMind latent extraction from TerraMesh S2RGB grid

This notebook runs a metadata-first pipeline for TerraMesh: query metadata with SQL, verify grid coverage, then download zarr tiles and save latent outputs to Parquet checkpoints.

In [ ]:
!pip install -q huggingface_hub datasets duckdb zarr xarray pyarrow geopandas shapely folium psutil
!pip install -q git+https://github.com/IBM/terratorch.git

In [ ]:
import gc
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import xarray as xr
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import clear_output, display
from terratorch.registry import FULL_MODEL_REGISTRY
from zarr.storage import ZipStore

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'Using device: {device}')

In [ ]:
REPO_ID = 'ibm-esa-geospatial/TerraMesh'
DATASET_SPLIT = 'train'

# Poland-like default area, configurable
BBOX = {
    'min_lat': 49.0,
    'max_lat': 55.0,
    'min_lon': 14.0,
    'max_lon': 24.0,
}

# Grid density is configurable; prompt emphasizes grid structure.
GRID_ROWS = 1000
GRID_COLS = 1000

# Set to an integer for shorter runs, or None for all selected cells.
MAX_SAMPLES = None

SAVE_EVERY = 1000
RESUME_FROM_CHECKPOINTS = True
INDICES = [0, 2, 4, 6, 8, 10, 12]

ENABLE_LIVE_DISPLAY = False
LIVE_REFRESH_EVERY = 25

OUTPUT_DIR = Path('outputs/latents_parquet')
HF_CACHE_DIR = Path('outputs/hf_cache')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PARQUET = OUTPUT_DIR / 'terramesh_train_metadata.parquet'

In [ ]:
input_modalities = ['S2RGB']
output_modalities = ['S2RGB']

model = FULL_MODEL_REGISTRY.build(
    'terramind_v1_tiny_generate',
    modalities=input_modalities,
    output_modalities=output_modalities,
    pretrained=True,
    standardize=True,
)
model = model.to(device)
model.eval()

def resolve_s2_embedding_key(model_obj: torch.nn.Module) -> str:
    keys = list(model_obj.sampler.model.encoder_embeddings.keys())
    if 'untok_s2rgb@224' in keys:
        return 'untok_s2rgb@224'
    if 'untok_sen2l2a@224' in keys:
        return 'untok_sen2l2a@224'
    for key in keys:
        key_l = key.lower()
        if 's2' in key_l and '@224' in key_l:
            return key
    raise KeyError(f'Could not resolve an S2 embedding key. Available keys: {keys}')

S2_EMBED_KEY = resolve_s2_embedding_key(model)
print(f'Using encoder embedding key: {S2_EMBED_KEY}')

def extract_terramind_latent(model_obj: torch.nn.Module, input_batch: torch.Tensor, indices: list[int]) -> list[torch.Tensor]:
    ordered_indices = sorted(indices)
    with torch.no_grad():
        emb_out = model_obj.sampler.model.encoder_embeddings[S2_EMBED_KEY](input_batch)
        x = emb_out['x'] if isinstance(emb_out, dict) else emb_out
        latents = []

        if 0 in ordered_indices:
            layer0 = model_obj.sampler.model.encoder_norm(x).mean(dim=1)[0].detach().cpu()
            latents.append(layer0)

        for layer_idx, block in enumerate(model_obj.sampler.model.encoder, start=1):
            x = block(x)
            if layer_idx in ordered_indices:
                latent = model_obj.sampler.model.encoder_norm(x).mean(dim=1)[0].detach().cpu()
                latents.append(latent)

    if len(latents) != len(ordered_indices):
        raise RuntimeError(
            f'Latent extraction mismatch: expected {len(ordered_indices)} layers, got {len(latents)}.'
        )
    return latents

In [ ]:
REQUIRED_COLUMNS = [
    'zarr', 'sample_id', 'split', 'center_lon', 'center_lat', 'crs',
    'bounds', 'geometry', 'S2_time', 'S1_time', 'cloud_cover', 'delta'
]

def materialize_terramesh_parquet(parquet_path: Path) -> None:
    print('Loading TerraMesh metadata from Hugging Face...')
    ds = load_dataset(REPO_ID, split=DATASET_SPLIT)
    ds.to_parquet(str(parquet_path), overwrite=True)
    print(f'Metadata parquet materialized at: {parquet_path}')

def query_terra_mesh(sql: str) -> pd.DataFrame:
    if not METADATA_PARQUET.exists():
        materialize_terramesh_parquet(METADATA_PARQUET)

    parquet_path_sql = str(METADATA_PARQUET).replace("'", "''")
    con = duckdb.connect(database=':memory:')
    try:
        con.execute(f"CREATE TABLE train AS SELECT * FROM read_parquet('{parquet_path_sql}')")
        return con.execute(sql).fetchdf()
    finally:
        con.close()

def fetch_candidate_metadata(bbox: dict[str, float]) -> pd.DataFrame:
    sql = f"""
    SELECT
        zarr,
        sample_id,
        split,
        center_lon,
        center_lat,
        crs,
        bounds,
        geometry,
        S2_time,
        S1_time,
        cloud_cover,
        delta
    FROM train
    WHERE center_lat BETWEEN {bbox['min_lat']} AND {bbox['max_lat']}
      AND center_lon BETWEEN {bbox['min_lon']} AND {bbox['max_lon']}
    ORDER BY center_lat DESC, center_lon ASC
    """
    return query_terra_mesh(sql)

def enforce_metadata_schema(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise KeyError(f'Missing required columns from SQL result: {missing}')

    out = df.copy()
    out['center_lon'] = pd.to_numeric(out['center_lon'], errors='raise').astype('float64')
    out['center_lat'] = pd.to_numeric(out['center_lat'], errors='raise').astype('float64')
    out['crs'] = pd.to_numeric(out['crs'], errors='raise').astype('int64')
    out['cloud_cover'] = pd.to_numeric(out['cloud_cover'], errors='raise').astype('float64')
    out['delta'] = pd.to_numeric(out['delta'], errors='raise').astype('float64')
    out['S2_time'] = pd.to_datetime(out['S2_time'], errors='raise')
    out['S1_time'] = pd.to_datetime(out['S1_time'], errors='raise')
    return out

def select_dense_grid(df: pd.DataFrame, bbox: dict[str, float], rows: int, cols: int, max_samples: int | None) -> pd.DataFrame:
    if df.empty:
        raise ValueError('No metadata rows returned for the configured BBOX.')

    lat_edges = np.linspace(bbox['min_lat'], bbox['max_lat'], rows + 1)
    lon_edges = np.linspace(bbox['min_lon'], bbox['max_lon'], cols + 1)

    out = df.copy()
    out['grid_row'] = np.clip(np.digitize(out['center_lat'], lat_edges, right=False) - 1, 0, rows - 1)
    out['grid_col'] = np.clip(np.digitize(out['center_lon'], lon_edges, right=False) - 1, 0, cols - 1)
    out['grid_id'] = out['grid_row'].astype(str) + '_' + out['grid_col'].astype(str)

    out['cell_center_lat'] = lat_edges[out['grid_row'].to_numpy()] + (lat_edges[1] - lat_edges[0]) / 2
    out['cell_center_lon'] = lon_edges[out['grid_col'].to_numpy()] + (lon_edges[1] - lon_edges[0]) / 2
    out['dist_to_cell_center'] = (
        (out['center_lat'] - out['cell_center_lat']) ** 2 +
        (out['center_lon'] - out['cell_center_lon']) ** 2
    )

    out = out.sort_values(
        by=['grid_id', 'dist_to_cell_center', 'center_lat', 'center_lon'],
        ascending=[True, True, False, True],
    )
    selected = out.drop_duplicates(subset='grid_id', keep='first')
    selected = selected.sort_values(['grid_row', 'grid_col']).reset_index(drop=True)

    if max_samples is not None:
        selected = selected.head(max_samples).copy()

    return selected

candidate_tiles = fetch_candidate_metadata(BBOX)
candidate_tiles = enforce_metadata_schema(candidate_tiles)
selected_tiles = select_dense_grid(candidate_tiles, BBOX, GRID_ROWS, GRID_COLS, MAX_SAMPLES)

print(f'Candidate metadata rows: {len(candidate_tiles):,}')
print(f'Selected grid rows: {len(selected_tiles):,}')
selected_tiles.head(3)

In [ ]:
def verify_grid_selection(raw_df: pd.DataFrame, selected_df: pd.DataFrame, rows: int, cols: int, bbox: dict[str, float]) -> dict[str, float]:
    occupancy = np.zeros((rows, cols), dtype=np.uint8)
    occupancy[selected_df['grid_row'].to_numpy(), selected_df['grid_col'].to_numpy()] = 1

    occupied = int(occupancy.sum())
    total_cells = rows * cols
    coverage_pct = 100.0 * occupied / total_cells

    print(f'Grid cells: {rows} x {cols} = {total_cells:,}')
    print(f'Occupied cells: {occupied:,}')
    print(f'Grid coverage: {coverage_pct:.4f}%')
    print(f'Candidate rows before selection: {len(raw_df):,}')
    print(f'Rows selected for extraction: {len(selected_df):,}')

    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    ax[0].imshow(occupancy, cmap='Greens', interpolation='nearest')
    ax[0].set_title('Grid occupancy (1=selected cell)')
    ax[0].set_xlabel('grid_col')
    ax[0].set_ylabel('grid_row')

    ax[1].scatter(selected_df['center_lon'], selected_df['center_lat'], s=1, alpha=0.8)
    ax[1].set_xlim([bbox['min_lon'], bbox['max_lon']])
    ax[1].set_ylim([bbox['min_lat'], bbox['max_lat']])
    ax[1].set_title('Selected tile centers')
    ax[1].set_xlabel('longitude')
    ax[1].set_ylabel('latitude')
    ax[1].grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()

    return {
        'grid_rows': rows,
        'grid_cols': cols,
        'total_cells': total_cells,
        'occupied_cells': occupied,
        'coverage_pct': coverage_pct,
        'selected_rows': len(selected_df),
    }

verification_report = verify_grid_selection(candidate_tiles, selected_tiles, GRID_ROWS, GRID_COLS, BBOX)
verification_report

In [ ]:
def download_zarr_archive(zarr_path: str) -> Path:
    local_file = hf_hub_download(
        repo_id=REPO_ID,
        filename=zarr_path,
        repo_type='dataset',
        cache_dir=str(HF_CACHE_DIR),
        force_download=False,
    )
    return Path(local_file)

def load_s2rgb_tensor(local_zip_path: Path) -> tuple[torch.Tensor, np.ndarray]:
    with ZipStore(str(local_zip_path), mode='r') as store:
        ds = xr.open_zarr(store, consolidated=False)
        available_vars = list(ds.data_vars)
        if 'S2RGB' not in ds:
            ds.close()
            raise KeyError(f"Expected 'S2RGB' variable in {local_zip_path}, found: {available_vars}")

        s2rgb = ds['S2RGB']
        arr = s2rgb.values
        ds.close()

    if arr.ndim == 4:
        arr = arr[0]
    if arr.ndim != 3:
        raise ValueError(f'Expected 3D S2RGB array, got shape {arr.shape} from {local_zip_path}')

    if arr.shape[0] == 3:
        chw = arr
    elif arr.shape[-1] == 3:
        chw = np.moveaxis(arr, -1, 0)
    else:
        raise ValueError(f'Cannot infer RGB channel axis for array shape {arr.shape}')

    chw = chw.astype(np.float32)
    if float(chw.max()) > 1.0:
        chw = np.clip(chw / 10000.0, 0.0, 1.0)
    else:
        chw = np.clip(chw, 0.0, 1.0)

    preview = np.moveaxis(chw, 0, -1)
    img = torch.from_numpy(chw)
    img = F.interpolate(
        img.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False
    )
    return img, preview

def normalize_bounds(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, tuple):
        return list(value)
    return value

def build_output_row(row: pd.Series, latents: list[torch.Tensor], indices: list[int]) -> dict:
    record = {
        'zarr': str(row['zarr']),
        'sample_id': str(row['sample_id']),
        'split': str(row['split']),
        'center_lon': float(row['center_lon']),
        'center_lat': float(row['center_lat']),
        'crs': int(row['crs']),
        'bounds': normalize_bounds(row['bounds']),
        'geometry': row['geometry'],
        'S2_time': pd.Timestamp(row['S2_time']),
        'S1_time': pd.Timestamp(row['S1_time']),
        'cloud_cover': float(row['cloud_cover']),
        'delta': float(row['delta']),
    }
    for idx, latent in zip(sorted(indices), latents):
        record[f'latent_layer_{idx}'] = latent.float().tolist()
    return record

def render_live_progress(current_rgb: np.ndarray, occupancy: np.ndarray, processed: int) -> None:
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(14, 6))
    ax[0].imshow(np.clip(current_rgb, 0.0, 1.0))
    ax[0].set_title(f'Current S2RGB sample #{processed}')
    ax[0].axis('off')

    ax[1].imshow(occupancy, cmap='Greens', interpolation='nearest')
    ax[1].set_title('Processed grid cells')
    ax[1].set_xlabel('grid_col')
    ax[1].set_ylabel('grid_row')

    plt.tight_layout()
    display(fig)
    plt.close(fig)

In [ ]:
if 'verification_report' not in globals():
    raise RuntimeError('Run metadata verification cell before downloading zarr files.')

checkpoint_paths = sorted(OUTPUT_DIR.glob('latents_checkpoint_*.parquet'))
already_processed = 0

if RESUME_FROM_CHECKPOINTS and checkpoint_paths:
    for path in checkpoint_paths:
        already_processed += len(pd.read_parquet(path, columns=['sample_id']))
    print(f'Resuming from {already_processed:,} already-processed rows across {len(checkpoint_paths)} checkpoint files.')
else:
    print('Starting from scratch.')

rows_to_process = selected_tiles.iloc[already_processed:].reset_index(drop=True)
print(f'Rows remaining in this run: {len(rows_to_process):,}')

buffer_rows = []
processed_total = already_processed
occupancy = np.zeros((GRID_ROWS, GRID_COLS), dtype=np.uint8)
if already_processed > 0:
    prefilled = selected_tiles.iloc[:already_processed]
    occupancy[prefilled['grid_row'].to_numpy(), prefilled['grid_col'].to_numpy()] = 1

for local_i, row in rows_to_process.iterrows():
    local_zip = download_zarr_archive(str(row['zarr']))
    image_tensor, preview = load_s2rgb_tensor(local_zip)
    image_tensor = image_tensor.to(device)
    latents = extract_terramind_latent(model, image_tensor, INDICES)

    out_row = build_output_row(row, latents, INDICES)
    buffer_rows.append(out_row)

    r = int(row['grid_row'])
    c = int(row['grid_col'])
    occupancy[r, c] = 1
    processed_total += 1

    if ENABLE_LIVE_DISPLAY and (processed_total % LIVE_REFRESH_EVERY == 0 or processed_total == 1):
        render_live_progress(preview, occupancy, processed_total)

    if processed_total % SAVE_EVERY == 0:
        checkpoint_path = OUTPUT_DIR / f'latents_checkpoint_{processed_total:09d}.parquet'
        pd.DataFrame(buffer_rows).to_parquet(checkpoint_path, index=False)
        print(f'Saved checkpoint: {checkpoint_path} ({len(buffer_rows):,} rows)')
        buffer_rows.clear()
        gc.collect()

if buffer_rows:
    checkpoint_path = OUTPUT_DIR / f'latents_checkpoint_{processed_total:09d}.parquet'
    pd.DataFrame(buffer_rows).to_parquet(checkpoint_path, index=False)
    print(f'Saved final checkpoint: {checkpoint_path} ({len(buffer_rows):,} rows)')
    buffer_rows.clear()

all_checkpoints = sorted(OUTPUT_DIR.glob('latents_checkpoint_*.parquet'))
manifest = pd.DataFrame({'checkpoint_file': [str(p) for p in all_checkpoints]})
manifest_path = OUTPUT_DIR / 'checkpoints_manifest.parquet'
manifest.to_parquet(manifest_path, index=False)

print(f'Total processed rows available: {processed_total:,}')
print(f'Checkpoint files: {len(all_checkpoints)}')
print(f'Manifest written to: {manifest_path}')

In [ ]:
# Optional: combine checkpoints into a single parquet file.
all_checkpoints = sorted(OUTPUT_DIR.glob('latents_checkpoint_*.parquet'))
if not all_checkpoints:
    raise FileNotFoundError('No checkpoint parquet files were found to combine.')

combined_df = pd.concat((pd.read_parquet(p) for p in all_checkpoints), ignore_index=True)
combined_path = OUTPUT_DIR / 'latents_full.parquet'
combined_df.to_parquet(combined_path, index=False)
print(f'Combined parquet saved to: {combined_path}')
print(f'Combined rows: {len(combined_df):,}')